In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT/CMS style
hep.style.use(hep.style.ROOT)

def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights**2)
    return counts, np.sqrt(variances)

def main():
    filename = "~/merged_elecs_test_tuples.root" 
    
    # 1. Bypass uproot! Load the tree using ROOT's C++ RDataFrame engine
    # This automatically uses ROOT's built-in file recovery if the file is slightly mangled.
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    # =========================================================================
    # APPLY STRICT PHYSICS CUTS AT THE C++ LEVEL (Super Fast!)
    # =========================================================================
    # Filter the DataFrame before pulling data into Python memory
    df_base = df.Filter("reco_mass >= 8.0 && reco_mass <= 12.0") \
                .Filter("abs(sigma_epz - 36.0) <= 3.6")

    # Standard mask (includes W cut)
    df_standard = df_base.Filter("reco_W >= 20.0")

    # =========================================================================
    # PULL DATA INTO PYTHON NUMPY ARRAYS
    # =========================================================================
    # AsNumpy() perfectly converts C++ branches into Python NumPy arrays
    cols_standard = df_standard.AsNumpy(["reco_mass", "reco_t_proj", "weight", "weight_proj", "reco_topology"])
    cols_w_plot   = df_base.AsNumpy(["reco_W", "weight"])

    mass = cols_standard["reco_mass"]
    wgt  = cols_standard["weight"]
    topo = cols_standard["reco_topology"]

    # =========================================================================
    # PLOT 1: INVARIANT MASS WITH COMBINED TOPOLOGY BREAKDOWN
    # =========================================================================
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    mass_bins = np.linspace(8.0, 12.0, 80)

    counts, errs = make_hist(mass, mass_bins, wgt)
    hep.histplot(counts, bins=mass_bins, yerr=errs, histtype='step', color='black', linewidth=2, label='Total Expected Yield')

    # Boolean masks in numpy work exactly the same as Awkward arrays
    mask_tagged_proton = (topo == 1) | (topo == 2)
    mask_missing_mass  = (topo == 3) | (topo == 4)

    if np.sum(mask_tagged_proton) > 0:
        counts_tag, _ = make_hist(mass[mask_tagged_proton], mass_bins, wgt[mask_tagged_proton])
        hep.histplot(counts_tag, bins=mass_bins, histtype='fill', alpha=0.4, color='blue', label='Combined: All Tagged Protons')
        hep.histplot(counts_tag, bins=mass_bins, histtype='step', color='darkblue', linewidth=1.5)

    if np.sum(mask_missing_mass) > 0:
        counts_miss, _ = make_hist(mass[mask_missing_mass], mass_bins, wgt[mask_missing_mass])
        hep.histplot(counts_miss, bins=mass_bins, histtype='fill', alpha=0.4, color='red', label='Combined: All Missing Mass')
        hep.histplot(counts_miss, bins=mass_bins, histtype='step', color='darkred', linewidth=1.5)

    sub_topologies = {
        1: ("Central $e^{-}$ (Tagged)", "blue", "--"),
        2: ("Low-$Q^2$ $e^{-}$ (Tagged)", "cyan", ":"),
        3: ("Central $e^{-}$ (Missing)", "red", "--"),
        4: ("Low-$Q^2$ $e^{-}$ (Missing)", "orange", ":")
    }

    for top_id, (label, color, linestyle) in sub_topologies.items():
        mask = (topo == top_id)
        if np.sum(mask) > 0:
            counts_sub, _ = make_hist(mass[mask], mass_bins, wgt[mask])
            hep.histplot(counts_sub, bins=mass_bins, histtype='step', color=color, linestyle=linestyle, linewidth=1.5, label=label)

    ax1.set_xlabel(r"Reconstructed $M_{e^{+}e^{-}}$ [GeV/$c^2$]")
    ax1.set_ylabel(r"Events / 50 MeV")
    ax1.set_title(r"$\Upsilon(1S) \rightarrow e^{+}e^{-}$ Photoproduction at EIC", loc="right", fontsize=16)
    hep.cms.text("Simulation Preliminary", loc=0)
    hep.cms.lumitext(r"$10$ fb$^{-1}$ (18x275 GeV)", ax=ax1)
    
    ax1.legend(loc="upper right", fontsize=11, ncol=1)
    ax1.set_xlim(8.0, 12.0)
    if np.max(counts) > 0: ax1.set_ylim(0, np.max(counts) * 1.4)

    fig1.tight_layout()
    fig1.savefig("Upsilon_Mass_CombinedTopologies.png", dpi=300)
    print("Saved Upsilon_Mass_CombinedTopologies.png")

    # =========================================================================
    # PLOT 2: PROJECTIVE t-SPECTRUM
    # =========================================================================
    fig2, ax2 = plt.subplots(figsize=(10, 8))

    t_proj   = cols_standard["reco_t_proj"]
    wgt_proj = cols_standard["weight_proj"]
    
    valid_t = (t_proj > 0)
    t_bins = np.linspace(0.0, 1.5, 30)

    counts_t, errs_t = make_hist(t_proj[valid_t], t_bins, wgt_proj[valid_t])

    hep.histplot(counts_t, bins=t_bins, yerr=errs_t, histtype='errorbar', color='black', 
                 marker='o', markersize=6, label=r'Projective $|t|$ (Wedge < $\pi/12$)')

    ax2.set_yscale('log')
    ax2.set_xlabel(r"$|t|$ [GeV$^2$]")
    ax2.set_ylabel(r"Events / 0.05 GeV$^2$")
    ax2.set_title(r"Reconstructed Momentum Transfer Squared", loc="right", fontsize=16)
    hep.cms.text("Simulation Preliminary", loc=0)
    hep.cms.lumitext(r"$10$ fb$^{-1}$ (18x275 GeV)", ax=ax2)
    
    ax2.legend(loc="upper right")
    ax2.set_xlim(0, 1.5)
    if np.sum(valid_t) > 0: ax2.set_ylim(bottom=0.5) 

    fig2.tight_layout()
    fig2.savefig("Upsilon_Projective_t.png", dpi=300)
    print("Saved Upsilon_Projective_t.png")

    # =========================================================================
    # PLOT 3: W SPECTRUM (Phase Space)
    # =========================================================================
    fig3, ax3 = plt.subplots(figsize=(10, 8))

    W_arr = cols_w_plot["reco_W"]
    W_wgt = cols_w_plot["weight"]
    
    valid_W_events = (W_arr > 0)
    W_bins = np.linspace(20, 150, 65)

    counts_w, _ = make_hist(W_arr[valid_W_events], W_bins, W_wgt[valid_W_events])

    hep.histplot(counts_w, bins=W_bins, histtype='fill', color='forestgreen', alpha=0.5, label='W Spectrum')
    hep.histplot(counts_w, bins=W_bins, histtype='step', color='darkgreen', linewidth=2)

    ax3.set_xlabel(r"$W_{\gamma p}$ [GeV]")
    ax3.set_ylabel(r"Events / 2 GeV")
    ax3.set_title(r"Photon-Proton Center of Mass Energy", loc="right", fontsize=16)
    hep.cms.text("Simulation Preliminary", loc=0)
    hep.cms.lumitext(r"$10$ fb$^{-1}$ (18x275 GeV)", ax=ax3)
    
    ax3.set_xlim(20, 150)
    if np.max(counts_w) > 0: ax3.set_ylim(0, np.max(counts_w) * 1.2)

    fig3.tight_layout()
    fig3.savefig("Upsilon_W_Spectrum.png", dpi=300)
    print("Saved Upsilon_W_Spectrum.png")

if __name__ == "__main__":
    main()